# Formula 1 Race Strategy Simulator
## Notebook 04: Real-World Telemetry Ingestion & Model Validation

This notebook covers **Phase 5**:
1. **Historical Telemetry Ingestion**: Loading OpenF1 2024 race weekend timing data.
2. **Clean Lap Filtering**: Eliminating pit in/out laps, standing starts, safety cars, and unrepresentative pace outliers ($107\%$ median filter).
3. **Fixed-Effects OLS Regression**: Correcting for fuel mass depletion and fitting compound degradation parameters $\alpha_c, \beta_c, \delta_c$.
4. **Driver Backtesting**: Quantitative validation against 2024 Grand Prix winners (Max Verstappen, Charles Leclerc).
5. **Discrepancy Diagnostics**: Engineering root-cause analysis of real-world tactical deviations.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_pipeline import load_historical_data
from src.estimation import estimate_empirical_parameters
from src.validation import RaceBacktester, DiscrepancyAnalyzer
from run_validation import DEFAULT_WINNERS

sns.set_theme(style="darkgrid")
print("Validation & telemetry pipeline initialized.")

### 1. Ingesting Real-World Telemetry

We load the historical 2024 Bahrain Grand Prix dataset, which contains complete telemetry records:
* Total recorded laps across all drivers
* Stint compound history
* Real pit stop transit and stationary times

In [ ]:
data = load_historical_data("bahrain")
print(f"Circuit: {data.circuit_name} ({data.year})")
print(f"Total Laps Recorded: {len(data.laps)}")
print(f"Total Pit Stops Recorded: {len(data.pit_stops)}")
print(f"Participating Drivers: {len(data.stints.driver_number.unique())}")

### 2. Empirical Parameter Estimation via OLS Regression

Raw lap times $T(n)$ mask degradation because fuel burn continuously accelerates the car:
$$T_{\text{corr}}(a) = T_{\text{raw}}(n) + \gamma_{\text{fuel}} \cdot (m_0 - m(n))$$

We fit a quadratic wear model for each tyre compound:
$$T_{\text{corr}}(a) = T_{base} + \delta_c + \alpha_c a + \beta_c a^2 + \epsilon$$

In [ ]:
emp_params = estimate_empirical_parameters(data)

print(f"Base Lap Time: {emp_params.base_lap_time:.3f} s")
print(f"Mean Pit Loss: {emp_params.pit_loss_mean:.2f} s (std: {emp_params.pit_loss_std:.2f} s)")

fitted_rows = []
for c_name, cp in emp_params.compounds.items():
    fitted_rows.append({
        "Compound": c_name,
        "Linear Alpha (s/lap)": f"{cp.alpha:.5f}",
        "Quadratic Beta (s/lap^2)": f"{cp.beta:.6f}",
        "Base Delta (s)": f"{cp.base_delta:+.3f}",
        "R-squared": f"{cp.r_squared:.3f}",
        "Clean Samples": cp.sample_count,
    })

print(pd.DataFrame(fitted_rows).to_string(index=False))

### 3. Backtesting Against 2024 Bahrain Winner: Max Verstappen

We test whether the calibrated simulator accurately reproduces the race-winning strategy of Max Verstappen (#1).

In [ ]:
backtester = RaceBacktester(data, empirical_params=emp_params)
driver_num, driver_name = DEFAULT_WINNERS["bahrain"]
metrics = backtester.backtest_driver(driver_num, driver_name=driver_name)

print("=== VERSTAPPEN BACKTEST METRICS ===")
print(f"Actual Strategy:              {metrics.actual_strategy_desc} ({metrics.actual_stops} stops)")
print(f"Simulated Optimal Strategy:   {metrics.optimal_strategy_desc} ({metrics.optimal_stops} stops)")
print(f"Stop Count Matched:           {metrics.stop_count_matches}")
print(f"Stint Length MAE:             {metrics.stint_length_mae:.2f} laps")
print(f"Clean Lap Pace RMSE:          {metrics.clean_lap_rmse:.3f} s")
print(f"Actual Race Duration:         {metrics.actual_time:.2f} s")
print(f"Simulated Actual Duration:    {metrics.sim_actual_strategy_time:.2f} s")
print(f"Physics Duration Error:       {metrics.sim_actual_error_pct:.3f}%")
print(f"Strategic Gain Delta:         {metrics.strategic_gain_delta:+.2f} s")

### 4. Residual Diagnostic Analysis

Let us inspect the distribution of lap-by-lap residuals $T_{\text{actual}} - T_{\text{simulated}}$.

In [ ]:
res_df = backtester.get_lap_residuals(driver_num)

plt.figure(figsize=(11, 4.5))
plt.subplot(1, 2, 1)
plt.scatter(res_df.lap_number, res_df.residual, color="royalblue", alpha=0.7, edgecolor="k")
plt.axhline(0, color="crimson", linestyle="--", linewidth=1.5)
plt.title("Lap Pace Residuals vs Race Lap", fontweight="bold")
plt.xlabel("Lap Number")
plt.ylabel("Residual: Actual - Simulated (s)")

plt.subplot(1, 2, 2)
sns.histplot(res_df.residual, kde=True, color="navy")
plt.axvline(0, color="crimson", linestyle="--")
plt.title("Residual Error Distribution", fontweight="bold")
plt.xlabel("Residual (s)")

plt.tight_layout()
plt.show()

### 5. Tactical Discrepancy Diagnostics

Why do real Formula 1 pit stop laps sometimes differ from analytical predictions?
The `DiscrepancyAnalyzer` categorizes tactical mechanisms:
* **Thermal tyre management**: Drivers delta-managing pace in early stint laps.
* **Dirty air mitigation**: Pitting into clean track windows rather than optimal wear points.
* **Undercut/Overcut tactics**: Tactical responses to rival pit window positioning.

In [ ]:
diag = DiscrepancyAnalyzer.analyze_circuit("bahrain")

print(f"Diagnostic Case: {diag.case_title}")
print(f"
Observed Phenomenon:
  {diag.observed_phenomenon}")
print(f"
Theoretical Model Prediction:
  {diag.theoretical_prediction}")
print("
Root Cause Factors:")
for f in diag.root_cause_factors:
    print(f"  * {f}")
print(f"
Engineering Solution:
  {diag.engineering_solution}")